# Patient Wait-Time Analysis

## Project Overview
Analyze healthcare operations data to understand how wait times vary by department, shift, patient type, and staffing.

## Learning Objectives
- Calculate wait-time distributions across clinical departments
- Identify peak hours and staffing-related bottlenecks
- Analyze the impact of patient acuity and visit type on wait times
- Quantify the relationship between staffing levels and wait times

## Problem Statement
Hospital operations needs to reduce patient wait times to improve satisfaction scores and clinical outcomes. They need data to identify when and where waits are longest and what drives them.

## Why This Project Matters
Long wait times reduce patient satisfaction, increase left-without-being-seen (LWBS) rates, and can delay time-sensitive treatments. Data-driven staffing optimization directly improves care quality.

## Dataset Overview
Synthetic patient visit dataset: ~8,000 visits across departments with timestamps, wait times, staffing levels, and patient characteristics.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 8000
departments = np.random.choice(['Emergency', 'Outpatient Clinic', 'Radiology', 'Lab', 'Pharmacy', 'Specialty'],
                                 n, p=[0.25, 0.25, 0.15, 0.15, 0.10, 0.10])
shift = np.random.choice(['Morning (7-15)', 'Afternoon (15-23)', 'Night (23-7)'], n, p=[0.45, 0.35, 0.20])
day_of_week = np.random.choice(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'],
                                 n, p=[0.17, 0.16, 0.15, 0.15, 0.15, 0.12, 0.10])
acuity = np.random.choice(['Critical', 'Urgent', 'Semi-Urgent', 'Non-Urgent'], n, p=[0.08, 0.22, 0.35, 0.35])
visit_type = np.random.choice(['Walk-In', 'Scheduled', 'Referral', 'Follow-Up'], n, p=[0.30, 0.35, 0.20, 0.15])
insurance = np.random.choice(['Medicare', 'Medicaid', 'Private', 'Self-Pay'], n, p=[0.30, 0.20, 0.38, 0.12])

# Staffing level (relative to ideal)
staffing_ratio = np.clip(np.random.normal(0.85, 0.15, n), 0.5, 1.2).round(2)

# Wait time driven by department, acuity, staffing
base_wait = {'Emergency': 35, 'Outpatient Clinic': 25, 'Radiology': 30,
             'Lab': 15, 'Pharmacy': 12, 'Specialty': 40}
acuity_mult = {'Critical': 0.3, 'Urgent': 0.6, 'Semi-Urgent': 1.0, 'Non-Urgent': 1.3}
shift_add = {'Morning (7-15)': 5, 'Afternoon (15-23)': 0, 'Night (23-7)': -5}

wait_min = np.array([base_wait[d] * acuity_mult[a] / staffing_ratio[i] + shift_add[s]
                      for i, (d, a, s) in enumerate(zip(departments, acuity, shift))])
wait_min = np.clip(wait_min + np.random.normal(0, 8, n), 2, 180).round(0).astype(int)

df = pd.DataFrame({
    'VisitID': [f'V{i:06d}' for i in range(n)],
    'Department': departments, 'Shift': shift, 'DayOfWeek': day_of_week,
    'Acuity': acuity, 'VisitType': visit_type, 'Insurance': insurance,
    'StaffingRatio': staffing_ratio, 'WaitMinutes': wait_min,
})

print(f'Dataset: {df.shape}')
print(f'Wait time stats:\n{df["WaitMinutes"].describe().round(1)}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nDepartment distribution:\n{df["Department"].value_counts()}')

## Overall Wait-Time Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['WaitMinutes'], bins=40, edgecolor='black', color='steelblue', alpha=0.7)
axes[0].axvline(df['WaitMinutes'].median(), color='red', linestyle='--',
                label=f'Median: {df["WaitMinutes"].median():.0f} min')
axes[0].set_title('Wait Time Distribution')
axes[0].set_xlabel('Minutes')
axes[0].legend()

sns.boxplot(data=df, x='Department', y='WaitMinutes', ax=axes[1])
axes[1].set_title('Wait Time by Department')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'wait_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()

## Wait Time by Shift and Day

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby('Shift')['WaitMinutes'].median().plot.bar(ax=axes[0], edgecolor='black', color='coral')
axes[0].set_title('Median Wait by Shift')
axes[0].tick_params(axis='x', rotation=0)

dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df.groupby('DayOfWeek')['WaitMinutes'].median().reindex(dow_order).plot.bar(ax=axes[1], edgecolor='black', color='teal')
axes[1].set_title('Median Wait by Day of Week')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'shift_day_wait.png'), dpi=100, bbox_inches='tight')
plt.show()

## Acuity and Visit Type Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

acuity_order = ['Critical', 'Urgent', 'Semi-Urgent', 'Non-Urgent']
sns.boxplot(data=df, x='Acuity', y='WaitMinutes', order=acuity_order, ax=axes[0])
axes[0].set_title('Wait Time by Acuity')

sns.boxplot(data=df, x='VisitType', y='WaitMinutes', ax=axes[1])
axes[1].set_title('Wait Time by Visit Type')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'acuity_visit.png'), dpi=100, bbox_inches='tight')
plt.show()

acuity_stats = df.groupby('Acuity')['WaitMinutes'].agg(['median', 'mean', 'std', 'count']).reindex(acuity_order).round(1)
print('Wait Time by Acuity:')
print(acuity_stats)

## Staffing Impact on Wait Times

In [ ]:
df['StaffBand'] = pd.cut(df['StaffingRatio'], bins=[0, 0.7, 0.85, 1.0, 1.5],
                          labels=['Critically Low', 'Below Target', 'At Target', 'Above Target'])
staff_wait = df.groupby('StaffBand')['WaitMinutes'].agg(['median', 'mean', 'count']).round(1)
print('Wait Time by Staffing Level:')
print(staff_wait)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(df['StaffingRatio'], df['WaitMinutes'], alpha=0.08, s=5, color='steelblue')
ax.set_xlabel('Staffing Ratio (vs Ideal)')
ax.set_ylabel('Wait Minutes')
ax.set_title('Staffing Ratio vs Wait Time')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'staffing_wait.png'), dpi=100, bbox_inches='tight')
plt.show()

## Long-Wait Analysis (>60 min)

In [ ]:
long_wait = df[df['WaitMinutes'] > 60]
print(f'Long waits (>60 min): {len(long_wait)} ({len(long_wait)/len(df):.1%})')
print(f'\nLong waits by Department:\n{long_wait["Department"].value_counts()}')
print(f'\nLong waits by Acuity:\n{long_wait["Acuity"].value_counts()}')
print(f'\nAvg staffing ratio for long waits: {long_wait["StaffingRatio"].mean():.2f} vs overall {df["StaffingRatio"].mean():.2f}')

## Interpretation and Business Insight
- **Specialty and Emergency** departments have the longest wait times — different root causes (complexity vs volume)
- **Morning shifts** have higher waits due to peak volume — staffing should be front-loaded
- **Critical patients** are triaged faster as expected, but **Non-Urgent** waits are unacceptably long in some departments
- **Staffing ratio** has a clear inverse relationship with wait time — understaffing directly drives longer waits
- **Walk-in** visits wait longer than scheduled visits — appointment availability reduces congestion
- **Long waits (>60 min)** cluster in understaffed shifts and high-volume departments

## Limitations
- Synthetic data with simplified wait-time drivers
- No provider-level variation data
- No patient flow or bottleneck queuing model
- No satisfaction or LWBS correlation
- No seasonal variation (flu season, holidays)

## How to Improve This Project
- Add queuing theory models for capacity planning
- Include provider-level productivity metrics
- Track LWBS rates and correlate with wait times
- Add patient satisfaction survey linkage
- Model optimal staffing schedules to minimize waits

## Production Considerations
- Real-time wait-time dashboards for charge nurses
- Automated staffing alerts when wait times exceed thresholds
- Patient-facing estimated wait-time displays
- Predictive volume forecasting for proactive staffing

## Common Mistakes
- Using mean instead of median for skewed wait-time distributions
- Not stratifying by acuity — mixing critical and non-urgent waits
- Ignoring staffing as a root cause and blaming demand alone
- Setting uniform wait-time targets across departments with different workflows

## Mini Challenge / Exercises
1. Calculate the % of Emergency visits where Critical patients waited >15 minutes.
2. What staffing ratio threshold keeps median wait below 30 minutes?
3. Estimate the additional staff needed to bring all departments below 25-min median wait.
4. Which Department × Shift combination has the worst wait times?

## Final Summary / Key Takeaways
- Wait-time analysis reveals operational bottlenecks and staffing gaps
- Staffing levels are the most actionable lever for reducing waits
- Acuity-based triage is working but non-urgent patients face excessive waits
- Shift-level and department-level analysis enables targeted scheduling improvements
- Real-time monitoring and predictive staffing are the path to sustained improvement